# ⚽ Notebook 01 — football-data.co.uk EDA

**Fuente:** `https://www.football-data.co.uk/mmz4281/2526/E0.csv`  
**Datos:** 291 partidos × 132 columnas | Temporada Premier League 2025/26  
**Guía de referencia:** `guia-ml1-premier-league-data.pdf` — Prof. Julián Zuluaga

Este notebook sigue la **Sección 3** de la guía.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Cargar datos procesados
m = pd.read_csv('../data/processed/matches_2526.csv', parse_dates=['Date'])
print(f'✅ {m.shape[0]} partidos × {m.shape[1]} columnas')
m.head(3)

## 1. Distribución de resultados

In [ ]:
vc = m['FTR'].value_counts()
print('Distribución de resultados (Full Time):')
for k, v in vc.items():
    label = {'H':'Home Win','D':'Draw','A':'Away Win'}[k]
    print(f'  {label:12s}: {v:4d}  ({v/len(m):.1%})')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Pie chart
axes[0].pie(vc.values, labels=['Home Win','Away Win','Draw'],
            colors=['#2196F3','#F44336','#9E9E9E'],
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('Resultados PL 2025/26')

# Goles por partido
axes[1].hist(m['TotalGoals'].dropna(), bins=range(0,10),
             color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(m['TotalGoals'].mean(), color='red', ls='--',
                label=f"Media: {m['TotalGoals'].mean():.2f}")
axes[1].set_title('Distribución de Goles por Partido')
axes[1].set_xlabel('Goles totales')
axes[1].legend()

# Over 2.5 / BTTS
stats = {'Over 2.5': m['Over2_5'].mean(), 'BTTS': m['BTTS'].mean()}
axes[2].bar(stats.keys(), [v*100 for v in stats.values()],
            color=['#4CAF50','#FF9800'], edgecolor='white')
for i, (k, v) in enumerate(stats.items()):
    axes[2].text(i, v*100+1, f'{v:.1%}', ha='center', fontweight='bold')
axes[2].set_ylabel('%')
axes[2].set_title('Stats de Apuestas')
axes[2].set_ylim(0, 80)

plt.tight_layout()
plt.savefig('../data/processed/plot_results_dist.png', dpi=150)
plt.show()

## 2. Correlaciones con GoalDiff (PDF p.7)

In [ ]:
# Features del PDF
features = ['SOTDiff', 'HST', 'ShotDiff', 'FoulDiff', 'CornerDiff',
            'HTHG', 'HTAG', 'HY', 'AY']
corr_df = m[features + ['GoalDiff']].corr()['GoalDiff'].drop('GoalDiff').sort_values(ascending=False)

colors = ['#4CAF50' if v > 0 else '#F44336' for v in corr_df.values]
plt.figure(figsize=(10, 5))
bars = plt.barh(corr_df.index, corr_df.values, color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, corr_df.values):
    plt.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.title('Correlación con GoalDiff (FTHG − FTAG)', fontsize=13)
plt.xlabel('Coeficiente de Pearson')
plt.tight_layout()
plt.savefig('../data/processed/plot_correlations.png', dpi=150)
plt.show()

print('\n📝 Interpretación (PDF p.7):')
print('  SOTDiff (+0.547): el predictor más fuerte')
print('  CornerDiff (+0.014): NO predice resultado')
print('  FoulDiff (−0.199): más faltas → pierde')

## 3. Odds como benchmark (PDF p.7)

In [ ]:
# Predicción usando Bet365 (casa favorita)
played = m[m['FTR'].notna() & m['B365H'].notna()].copy()
for s in ['H', 'D', 'A']:
    played[f'Prob365{s}'] = 1 / pd.to_numeric(played[f'B365{s}'], errors='coerce')

played['bet365_pred'] = played[['Prob365H','Prob365D','Prob365A']].idxmax(axis=1)
played['bet365_pred'] = played['bet365_pred'].str.replace('Prob365', '')

acc_3class = (played['bet365_pred'] == played['FTR']).mean()
# Sin empates
no_draw = played[played['FTR'] != 'D'].copy()
acc_nodraw = (no_draw['bet365_pred'] == no_draw['FTR']).mean()
baseline_home = (played['FTR'] == 'H').mean()

print('📊 Benchmarks (PDF p.7):')
print(f'  Bet365 accuracy (3 clases H/D/A):   {acc_3class:.1%}')
print(f'  Bet365 accuracy (sin empates):       {acc_nodraw:.1%}')
print(f'  Baseline naive (siempre Home Win):   {baseline_home:.1%}')
print(f'  → Un modelo ML debe superar {acc_3class:.1%} en 3 clases')

## 4. Análisis por equipo

In [ ]:
ts = pd.read_csv('../data/processed/team_stats_2526.csv')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Goles anotados vs recibidos
ts_sorted = ts.sort_values('total_gf', ascending=True)
y = range(len(ts_sorted))
axes[0].barh(y, ts_sorted['total_gf'], color='#4CAF50', alpha=0.8, label='GF')
axes[0].barh(y, -ts_sorted['total_ga'], color='#F44336', alpha=0.8, label='GA')
axes[0].set_yticks(list(y)); axes[0].set_yticklabels(ts_sorted['team'], fontsize=8)
axes[0].axvline(0, color='black', linewidth=0.5)
axes[0].set_title('Goles Anotados vs Recibidos', fontsize=12)
axes[0].legend()

# Goal Difference
ts_gd = ts.sort_values('total_gd', ascending=True)
colors_gd = ['#4CAF50' if v >= 0 else '#F44336' for v in ts_gd['total_gd']]
axes[1].barh(range(len(ts_gd)), ts_gd['total_gd'], color=colors_gd, edgecolor='white')
axes[1].set_yticks(range(len(ts_gd))); axes[1].set_yticklabels(ts_gd['team'], fontsize=8)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Diferencia de Goles por Equipo', fontsize=12)

plt.suptitle('Premier League 2025/26 — Estadísticas por Equipo', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/plot_team_stats.png', dpi=150)
plt.show()

## 5. Preparación del feature set para ML

In [ ]:
# Feature engineering para clasificación (FTR)
ml_features = ['HS','AS','HST','AST','HF','AF','HC','AC','HY','AY',
                'ShotDiff','SOTDiff','FoulDiff','CornerDiff',
                'HTHG','HTAG']

ml_df = m[ml_features + ['FTR','GoalDiff','Over2_5','BTTS']].dropna()
print(f'✅ Dataset para ML: {ml_df.shape}')
print(f'   Features: {len(ml_features)}')
print(f'   Distribución objetivo FTR:')
print(f'   {ml_df["FTR"].value_counts().to_dict()}')

ml_df.to_csv('../data/processed/ml_ready_matches.csv', index=False)
print('\n💾 Guardado en data/processed/ml_ready_matches.csv')